# Imagined dragon sounds

Generate 16 distinct dragon sound effects with Stable Audio Open 1.0. Each uses a different description, seed and duration (3–10.5 seconds).

The prompts and settings below match the downloaded `sound-generation-log.json`. This notebook was aligned with that log after generation; execution outputs have not been reconstructed. The original Colab library versions were not recorded.

Use a GPU runtime. Review the access conditions at https://huggingface.co/stabilityai/stable-audio-open-1.0 using your Hugging Face account. Store a read token in Colab Secrets as HF_TOKEN and enable Notebook access. No token is included in this repository.


In [ ]:
!pip uninstall -y torchao
!pip install -q -U diffusers torchsde transformers accelerate soundfile sentencepiece


In [ ]:
import torch, soundfile as sf, shutil, json
import numpy as np
from pathlib import Path
from huggingface_hub import login
from diffusers import StableAudioPipeline
from google.colab import files, userdata

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU is connected. In Colab choose Runtime > Change runtime type > T4 GPU, '
        'save, reconnect, and rerun the installation cell. A Hugging Face token does not enable a GPU.'
    )
print('GPU connected:', torch.cuda.get_device_name(0))

SECRET_NAME = 'HF_TOKEN'  # Must match the name in Colab Secrets.
try:
    hf_token = userdata.get(SECRET_NAME)
except Exception:
    raise RuntimeError(
        'Cannot read HF_TOKEN. In the left-hand Secrets panel, name your Hugging Face '
        'read token HF_TOKEN and enable Notebook access for this notebook.'
    ) from None
if not hf_token:
    raise RuntimeError('The HF_TOKEN secret is empty.')
login(token=hf_token, add_to_git_credential=False)
del hf_token
print('Hugging Face login complete.')
pipe = StableAudioPipeline.from_pretrained(
    'stabilityai/stable-audio-open-1.0', torch_dtype=torch.float16
).to('cuda')
print('Audio model loaded successfully.')


In [ ]:
# These 16 prompts and settings match the downloaded generation log.
settings = [
    {
        "filename": "dragon-01.wav",
        "prompt": "A huge dragon giving a deep throaty roar, layered animal growls, powerful chest rumble, isolated creature sound effect",
        "seed": 42,
        "requested_duration_seconds": 3.0,
        "saved_duration_seconds": 3.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-02.wav",
        "prompt": "A small baby dragon making playful high pitched chirps and squeaks, short curious animal calls, close recording",
        "seed": 137,
        "requested_duration_seconds": 3.5,
        "saved_duration_seconds": 3.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-03.wav",
        "prompt": "An angry dragon hissing like a giant reptile, rough rasping breath followed by a sharp snarl, isolated sound effect",
        "seed": 256,
        "requested_duration_seconds": 4.0,
        "saved_duration_seconds": 4.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-04.wav",
        "prompt": "A sleepy dragon purring softly, slow rhythmic breathing, warm low vibrating animal rumble",
        "seed": 389,
        "requested_duration_seconds": 4.5,
        "saved_duration_seconds": 4.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-05.wav",
        "prompt": "A dragon breathing fire, forceful rushing exhalation, a burst of flame and crackling embers, cinematic sound effect",
        "seed": 512,
        "requested_duration_seconds": 5.0,
        "saved_duration_seconds": 5.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-06.wav",
        "prompt": "A distant dragon calling across a mountain valley, long low animal howl with a natural echo",
        "seed": 678,
        "requested_duration_seconds": 5.5,
        "saved_duration_seconds": 5.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-07.wav",
        "prompt": "A water dragon making deep bubbling growls and hollow gurgles, wet amphibian vocalizations beside a stream",
        "seed": 731,
        "requested_duration_seconds": 6.0,
        "saved_duration_seconds": 6.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-08.wav",
        "prompt": "A forest dragon communicating with rapid tongue clicks and short birdlike trills, clear animal recording",
        "seed": 864,
        "requested_duration_seconds": 6.5,
        "saved_duration_seconds": 6.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-09.wav",
        "prompt": "A wounded dragon giving a mournful descending whine, rasping breath and a soft low groan",
        "seed": 999,
        "requested_duration_seconds": 7.0,
        "saved_duration_seconds": 7.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-10.wav",
        "prompt": "A dragon giving three sharp warning barks, harsh reptilian cries followed by a low growl",
        "seed": 1024,
        "requested_duration_seconds": 7.5,
        "saved_duration_seconds": 7.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-11.wav",
        "prompt": "A mother dragon making gentle cooing calls to her young, soft resonant animal murmurs and purrs",
        "seed": 1187,
        "requested_duration_seconds": 8.0,
        "saved_duration_seconds": 8.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-12.wav",
        "prompt": "An ice dragon making a thin piercing whistle, airy breath and a brittle rasping animal cry",
        "seed": 1314,
        "requested_duration_seconds": 8.5,
        "saved_duration_seconds": 8.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-13.wav",
        "prompt": "A territorial dragon producing a long powerful bellow, deep crocodile-like growl layered with a lion roar",
        "seed": 1456,
        "requested_duration_seconds": 9.0,
        "saved_duration_seconds": 9.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-14.wav",
        "prompt": "A young dragon chattering excitedly, a sequence of squeaks, clicks and rising trills",
        "seed": 1601,
        "requested_duration_seconds": 9.5,
        "saved_duration_seconds": 9.5,
        "inference_steps": 100
    },
    {
        "filename": "dragon-15.wav",
        "prompt": "A dragon making a haunting melodic call, hollow resonant animal whistles with a gentle forest echo",
        "seed": 1777,
        "requested_duration_seconds": 10.0,
        "saved_duration_seconds": 10.0,
        "inference_steps": 100
    },
    {
        "filename": "dragon-16.wav",
        "prompt": "An ancient dragon exhaling slowly and making a very deep gravelly rumble that fades into silence",
        "seed": 2026,
        "requested_duration_seconds": 10.5,
        "saved_duration_seconds": 10.5,
        "inference_steps": 100
    }
]

from IPython.display import Audio, display
output_dir = Path('/content/dragon_sounds/sounds')
output_dir.mkdir(parents=True, exist_ok=True)
sr = pipe.vae.sampling_rate
generation_log = []
for row in settings:
    generator = torch.Generator('cuda').manual_seed(row['seed'])
    audio = pipe(
        row['prompt'], negative_prompt='music, EDM, instruments, human speech, singing, low quality, distortion, clipping',
        num_inference_steps=row['inference_steps'],
        audio_end_in_s=row['requested_duration_seconds'],
        num_waveforms_per_prompt=1, generator=generator
    ).audios
    waveform = audio[0].T.float().cpu().numpy()
    waveform = waveform[:round(row['requested_duration_seconds'] * sr)]
    filepath = output_dir / row['filename']
    sf.write(filepath, waveform, sr, subtype='PCM_16')
    generation_log.append({**row, 'saved_duration_seconds': len(waveform)/sr})
    print(row['filename'], len(waveform)/sr, 'seconds')
    display(Audio(filename=str(filepath)))
    del audio, waveform
    torch.cuda.empty_cache()
(output_dir.parent/'sound-generation-log.json').write_text(json.dumps(generation_log, indent=2))


In [ ]:
import zipfile
zip_path = Path('/content/dragon-sounds.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filepath in sorted(output_dir.glob('dragon-*.wav')):
        archive.write(filepath, 'sounds/'+filepath.name)
    archive.write(output_dir.parent/'sound-generation-log.json', 'sound-generation-log.json')
files.download(str(zip_path))
